# EDA — v2 text-ready subset (videos with title + description + Politics topic)

[![Python 3.10+](https://img.shields.io/badge/python-3.10+-blue.svg)](https://www.python.org/)
[![Jupyter](https://img.shields.io/badge/notebook-jupyter-orange.svg)](https://jupyter.org/)

---

## Purpose

This notebook is the analytical companion to **EXPORT 1** of the v2 data
extraction (see project README). The export selects every video that has

1. a `TITLE` and a `DESCRIPTION` captured at least once during the
   2024-01-01 to 2024-02-24 metadata-fetch window, **and**
2. an associated YouTube auto-classification topic of *Politics*
   (Wikipedia URL `Politics`).

The result is **21,710 videos** — the subset that is ready for the v2
semantic-embedding stage (multilingual encoder over title + description).
This notebook profiles the subset for quality, language, temporal and
content characteristics, and ends with concrete recommendations for the
embedding stage.

## Table of contents

1. [Setup and load](#1-setup-and-load)
2. [Coverage and basic shape](#2-coverage)
3. [Text length distributions](#3-text-length)
4. [Duration profile (shorts vs long-form)](#4-duration)
5. [Live broadcast status](#5-live)
6. [Language profile (declared vs audio)](#6-language)
7. [Temporal profile of publication dates](#7-temporal)
8. [Hashtag structure](#8-hashtags)
9. [URL and domain analysis](#9-urls)
10. [Description boilerplate / channel-proxy clusters](#10-boilerplate)
11. [Script mixing (Cyrillic / Latin)](#11-scripts)
12. [Protest keyword signal in title vs description](#12-protest-signal)
13. [Country signal (BY / UA / RU)](#13-country)
14. [Content-quality assessment](#14-quality)
15. [Embedding readiness — recommendation](#15-embedding-readiness)
16. [Findings and next steps](#16-findings)


<a id="1-setup-and-load"></a>
## 1. Setup and load

Minimal scientific-Python stack. The export is a comma-separated CSV with
double-quote enclosure and embedded newlines inside `DESCRIPTION` — pandas
handles this without any special arguments.


In [ ]:
import os
import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

DATA_PATH = os.environ.get(
    "PROTEST_EXPORT_PATH",
    "_All_videos_that_have_BOTH_title_description_AND_politics_topic__202605041410.csv",
)

df = pd.read_csv(DATA_PATH, dtype=str, keep_default_na=False, na_values=[""])
df["DESCRIPTION"] = df["DESCRIPTION"].fillna("")
df["TITLE"]       = df["TITLE"].fillna("")

print(f"Rows: {len(df):,}  |  Columns: {list(df.columns)}")


<a id="2-coverage"></a>
## 2. Coverage and basic shape

The export is the intersection of two filters: `VIDEO_METADATA` (which
covers only 27,305 videos out of the 401,105 in the corpus, since title
and description were captured in a single 46-day window in 2024) and
`VID_TOPICS = Politics` (which covers 70.5% of all videos). The
intersection is **21,710 videos** — small relative to the corpus but
large enough for embedding-based work.


In [ ]:
print(f"Total rows: {len(df):,}")
print(f"Unique YT_IDs: {df['YT_ID'].nunique():,}\n")
print("Null counts and percentages:")
nulls = df.isna().sum()
for col, n in nulls.items():
    print(f"  {col:25s} null={n:6,}  ({100*n/len(df):5.1f}%)")


<a id="3-text-length"></a>
## 3. Text length distributions

Title length is bounded by YouTube's 100-character limit, and
descriptions by the 5,000-character limit. The export confirms both:
99% of titles are at most 100 characters (the rare 101-character cases
appear to be edge-case Unicode counts), and 99% of descriptions are at
most ~4,800 characters. Median description length is ~900 characters,
which is comfortable for a single forward pass through a sentence
encoder limited to 512 tokens.


In [ ]:
df["title_len"] = df["TITLE"].str.len()
df["desc_len"]  = df["DESCRIPTION"].str.len()

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].hist(df["title_len"], bins=50, color="#3a6ea5", edgecolor="white")
ax[0].set_title("Title length (chars)")
ax[0].set_xlabel("characters")
ax[1].hist(df["desc_len"].clip(upper=5000), bins=60, color="#5a8e3a", edgecolor="white")
ax[1].set_title("Description length (chars; clipped at 5000)")
ax[1].set_xlabel("characters")
plt.tight_layout(); plt.show()

print("\nSummary:")
print(df[["title_len","desc_len"]].describe(percentiles=[.1,.25,.5,.75,.9,.99]).round(1))


<a id="4-duration"></a>
## 4. Duration profile (shorts vs long-form)

`DURATION` is stored in ISO 8601 format (`PT24S`, `PT1M22S`, `PT2H30M0S`).
We parse it to seconds and bucket. The corpus is heavily mid-form: ~30%
falls into the 5–20 minute range, with substantial Shorts (<60 s)
representation (~22%). The 1h+ tail is entirely long-form interviews
and live-stream archives.


In [ ]:
def parse_dur(s):
    if not s: return None
    m = re.match(r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?$", s)
    if not m: return None
    h, mi, se = (int(x) if x else 0 for x in m.groups())
    return h*3600 + mi*60 + se

df["dur_sec"] = df["DURATION"].map(parse_dur)
print(f"Unparsed durations: {df['dur_sec'].isna().sum()} (likely live or upcoming)")

bins   = [0, 60, 300, 1200, 3600, 86400]
labels = ["<60s (Shorts)", "1–5 min", "5–20 min", "20–60 min", "1h+"]
df["dur_bucket"] = pd.cut(df["dur_sec"], bins=bins, labels=labels)

ax = df["dur_bucket"].value_counts().reindex(labels).plot.bar(
    color="#3a6ea5", figsize=(8, 3.5), edgecolor="white")
ax.set_title("Duration distribution")
ax.set_ylabel("videos"); ax.tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()
print(df["dur_bucket"].value_counts().reindex(labels).to_string())


<a id="5-live"></a>
## 5. Live broadcast status

`LIVE_BROADCAST` distinguishes regular uploads from live-stream
archives. The 246 archived live broadcasts are likely event coverage
(rallies, press conferences) and warrant inspection as a distinct
sub-category — they are typically much longer and contain different
descriptive metadata.


In [ ]:
print(df['LIVE_BROADCAST'].value_counts().to_string())


<a id="6-language"></a>
## 6. Language profile

Two language fields are reported: `DEFAULT_LANG` (declared by the
uploader, often unset) and `DEFAULT_AUDIO_LANG` (more reliably
populated). Russian dominates at ~43% of audio-language declarations,
with English second. Belarusian is rare (~2%) — consistent with the
sociolinguistic dominance of Russian in Belarusian online discourse,
not with absence of Belarusian-context content.

> **Implication for the embedding stage.** A multilingual encoder
> (`intfloat/multilingual-e5-large` or `BAAI/bge-m3`) is required.
> Encoders trained primarily on English will significantly
> underperform on the 80% non-English subset.


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))

dl = df["DEFAULT_LANG"].fillna("(unset)").value_counts().head(12)
dal = df["DEFAULT_AUDIO_LANG"].fillna("(unset)").value_counts().head(12)

dl.plot.bar(ax=ax[0], color="#3a6ea5", edgecolor="white")
ax[0].set_title("DEFAULT_LANG (declared)")
ax[0].tick_params(axis='x', rotation=45)

dal.plot.bar(ax=ax[1], color="#5a8e3a", edgecolor="white")
ax[1].set_title("DEFAULT_AUDIO_LANG (more reliable)")
ax[1].tick_params(axis='x', rotation=45)

plt.tight_layout(); plt.show()


<a id="7-temporal"></a>
## 7. Temporal profile of publication dates

The PUBLISHED_AT distribution shows the data-collection skew very
clearly. Even though the corpus reaches back to 2007, **57% of the
text-ready subset was published in 2024 alone** (12,355 videos), with
2023 a distant second (5,037). The 2020 Belarusian-protest year accounts
for only 939 videos, and the 2014 Euromaidan year only 30 — confirming
the recency-bias finding from the manuscript.

> **Implication.** Any politically-motivated content analysis on this
> subset is implicitly *2024-centric*. To examine earlier protest events
> at scale, additional metadata fetches (or a second collection wave
> targeting earlier search-result archives) are required.


In [ ]:
df["pub_year"] = pd.to_datetime(df["PUBLISHED_AT"], errors="coerce").dt.year
yearly = df["pub_year"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 3.8))
yearly.plot.bar(ax=ax, color="#3a6ea5", edgecolor="white")
ax.set_title("Videos by publication year (2024 dominates)")
ax.set_ylabel("videos")
ax.tick_params(axis='x', rotation=0)
for i, v in enumerate(yearly.values):
    ax.text(i, v + 100, f"{v:,}", ha="center", fontsize=8)
plt.tight_layout(); plt.show()


In [ ]:
# Month-level density (2020 onwards)
df["pub_dt"] = pd.to_datetime(df["PUBLISHED_AT"], errors="coerce")
recent = df[df["pub_dt"] >= "2020-01-01"]
monthly = recent["pub_dt"].dt.to_period("M").value_counts().sort_index()
monthly.index = monthly.index.astype(str)

fig, ax = plt.subplots(figsize=(13, 3.5))
ax.plot(monthly.index, monthly.values, color="#3a6ea5")
ax.set_title("Monthly density of new political videos (2020 →)")
ax.set_ylabel("videos / month")
step = max(1, len(monthly) // 12)
ax.set_xticks(monthly.index[::step]); ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()


<a id="8-hashtags"></a>
## 8. Hashtag structure

YouTube hashtag usage in this subset is bimodal: most titles contain
none, but a subset of news-channel and shorts uploads use 5+. The most
frequent hashtags are predictable thematic anchors (`#новости`, `#shorts`,
`#ukraine`, `#putin`) but also reveal channel-specific brands
(`#freeдом`, `#kanal13`) that group videos better than generic tags.


In [ ]:
hashtag_re = re.compile(r"#([\wЀ-ӿ]+)", re.UNICODE)
df["title_hashtags"] = df["TITLE"].apply(hashtag_re.findall)
df["desc_hashtags"]  = df["DESCRIPTION"].apply(hashtag_re.findall)
df["n_title_tags"]   = df["title_hashtags"].str.len()
df["n_desc_tags"]    = df["desc_hashtags"].str.len()

print(f"Titles with ≥1 hashtag: {(df.n_title_tags >= 1).sum():,} ({100*(df.n_title_tags >= 1).mean():.1f}%)")
print(f"Median hashtags per description: {int(df.n_desc_tags.median())}")
print(f"Mean hashtags per description:   {df.n_desc_tags.mean():.1f}")

all_hashtags = Counter()
for hs in df["title_hashtags"]: all_hashtags.update(h.lower() for h in hs)
for hs in df["desc_hashtags"]:  all_hashtags.update(h.lower() for h in hs)

top = pd.Series(dict(all_hashtags.most_common(25)))
fig, ax = plt.subplots(figsize=(8, 6))
top.iloc[::-1].plot.barh(ax=ax, color="#3a6ea5", edgecolor="white")
ax.set_title("Top 25 hashtags (title + description, lowercased)")
plt.tight_layout(); plt.show()


<a id="9-urls"></a>
## 9. URL and domain analysis

Descriptions are URL-heavy: 90% of videos have at least 1 URL, the
median is 5, and 1% have 35+. The top domain is `bit.ly` — a URL
shortener that hides the actual destination — followed by `youtube.com`
self-references (sub_confirmation, channel pages), then social-media
cross-posting domains (`t.me`, `facebook.com`, `instagram.com`,
`twitter.com`). Several Belarusian and Ukrainian opposition outlets
appear as their own domains (`belsat.eu`, `vot-tak.tv`, `uatv.ua`).

> **Implication for embedding.** URLs add noise to the encoder input
> without conveying semantic information. We will replace them with a
> `<URL>` placeholder before encoding (preserves the *fact* of a link
> without forcing the model to embed arbitrary subdomains).


In [ ]:
url_re    = re.compile(r"https?://[^\s<>\"']+", re.UNICODE)
domain_re = re.compile(r"https?://(?:www\.)?([^\s/<>\"']+)", re.UNICODE)

df["n_urls"] = df["DESCRIPTION"].apply(lambda s: len(url_re.findall(s)))

domains = Counter()
for d in df["DESCRIPTION"]:
    for m in domain_re.findall(d):
        domains[m.lower()] += 1

top_domains = pd.Series(dict(domains.most_common(20)))
fig, ax = plt.subplots(figsize=(9, 5.5))
top_domains.iloc[::-1].plot.barh(ax=ax, color="#5a8e3a", edgecolor="white")
ax.set_title("Top 20 domains in descriptions")
plt.tight_layout(); plt.show()

print("URL count per description:")
print(df["n_urls"].describe(percentiles=[.1,.5,.9,.99]).round(1).to_string())


<a id="10-boilerplate"></a>
## 10. Description boilerplate / channel-proxy clusters

The first 200 characters of each description function as a fingerprint.
612 such heads appear on 2+ videos, and 40 appear on 10+ — strong
evidence that channels reuse boilerplate ("Подписывайтесь на канал...",
"#Kanal13 #likekanal13..."). This is a **free channel-id proxy**: videos
sharing a description-head almost certainly share a channel.


In [ ]:
df["desc_head"] = df["DESCRIPTION"].str[:200].str.replace(r"\s+", " ", regex=True).str.strip()
heads = df["desc_head"].value_counts()

print(f"Unique 200-char description heads: {len(heads):,}")
print(f"Heads appearing 2+ videos: {(heads >= 2).sum():,}")
print(f"Heads appearing 10+ videos: {(heads >= 10).sum():,}\n")

print("Top 10 most-replicated description heads (channel-proxy clusters):")
for head, n in heads.head(15).items():
    if n < 2 or len(head) < 5: continue
    print(f"  ({n:4d}×) {head[:110]!r}")


<a id="11-scripts"></a>
## 11. Script mixing (Cyrillic / Latin)

About 12% of titles mix Cyrillic and Latin scripts in the same string —
typically a Russian/Ukrainian title with English hashtags or a
transliterated proper noun ("Putin сказал...", "Lukashenka не уйдёт").
This is one reason monolingual encoders fail on this corpus.


In [ ]:
def has_cyr(s): return bool(re.search(r"[А-ЯЁЪа-яёъ]", s))
def has_lat(s): return bool(re.search(r"[A-Za-z]", s))

df["t_cyr"] = df["TITLE"].apply(has_cyr)
df["t_lat"] = df["TITLE"].apply(has_lat)

cnt = pd.Series({
    "Cyrillic only":  ((df.t_cyr) & (~df.t_lat)).sum(),
    "Latin only":     ((~df.t_cyr) & (df.t_lat)).sum(),
    "Mixed":          ((df.t_cyr) & (df.t_lat)).sum(),
    "Neither":        ((~df.t_cyr) & (~df.t_lat)).sum(),
})
fig, ax = plt.subplots(figsize=(7, 3))
cnt.plot.barh(ax=ax, color=["#3a6ea5","#5a8e3a","#a55e3a","#999"], edgecolor="white")
ax.set_title("Title scripts")
plt.tight_layout(); plt.show()
print(cnt.to_string())


<a id="12-protest-signal"></a>
## 12. Protest keyword signal in title vs description

A multilingual regex over protest-root vocabulary
(`протест\w*`, `protest\w*`, `митинг\w*`, `пратэст\w*`, `revolution`,
`maidan`, etc.) finds **9.4% of the subset (2,034 videos) mention
protest somewhere in title or description**. Of these:

- 156 videos mention protest *only in the title*
- 1,366 videos mention protest *only in the description* — these are
  the most interesting: protest content the title doesn't advertise
- 512 mention it in both
- The remaining 19,676 videos have no surface protest keyword — they
  are political content that talks *around* protest (parliamentary
  debates, sanctions, etc.)

This is the polysemy problem in microcosm: the keyword `протест` covers
only ~1 in 11 videos in this Politics-topic subset, while the rest are
politically adjacent.


In [ ]:
protest_pat = re.compile(
    r"(\bпротест\w*|\bprotest\w*|\bмитинг\w*|\bпратэст\w*|\bмайдан|maidan|"
    r"\bреволюц\w*|revolution|демонстра|demonstration|rally|rallies|"
    r"riot|восстан\w*|забастовк\w*|стачк\w*|stachk\w*|бунт\w*)",
    re.IGNORECASE | re.UNICODE,
)

df["protest_in_title"] = df["TITLE"].apply(lambda s: bool(protest_pat.search(s)))
df["protest_in_desc"]  = df["DESCRIPTION"].apply(lambda s: bool(protest_pat.search(s)))
df["protest_anywhere"] = df["protest_in_title"] | df["protest_in_desc"]

t_only = (df.protest_in_title & ~df.protest_in_desc).sum()
d_only = (~df.protest_in_title & df.protest_in_desc).sum()
both   = (df.protest_in_title & df.protest_in_desc).sum()
none_  = (~df.protest_in_title & ~df.protest_in_desc).sum()

print(f"Protest signal anywhere: {df.protest_anywhere.sum():,} / {len(df):,}  ({100*df.protest_anywhere.mean():.1f}%)")
print(f"  in TITLE only:        {t_only:,}")
print(f"  in DESCRIPTION only:  {d_only:,}")
print(f"  in BOTH:              {both:,}")
print(f"  in NEITHER:           {none_:,}\n")

# Venn-style
fig, ax = plt.subplots(figsize=(5,3))
ax.bar(["Title only","Desc only","Both","Neither"],
       [t_only, d_only, both, none_],
       color=["#3a6ea5","#5a8e3a","#a55e3a","#cccccc"], edgecolor="white")
ax.set_title("Where the protest keyword appears")
ax.set_ylabel("videos"); ax.set_yscale("log")
plt.tight_layout(); plt.show()


<a id="13-country"></a>
## 13. Country signal (BY / UA / RU)

Each video is labelled with the country contexts mentioned anywhere in
its title or description. Three patterns to note:

- The most common single context is **R** (Russia-only) — about 25% of
  the subset.
- The next is **U+R** (Ukraine + Russia, ~23%) — these are the war-
  coverage videos that mention both sides, which is the bulk of 2022-2024
  political content.
- **B+U+R** (all three, ~5%) is rarer than expected — most videos focus
  on one or two contexts at a time.
- 1,040 videos (~5%) mention none of the three — these are tangentially
  political content that the YouTube classifier flagged but which
  doesn't anchor to the Eastern Slavic frame.

This combination column is the **most useful proxy for polysemy
disambiguation**: a tag like `протест` carries different meaning in a
B-only video versus an R-only video, regardless of co-occurring tags.


In [ ]:
cs = {
    "BY": re.compile(r"(\bбеларус\w*|\bbelarus\w*|\bbelarusian\w*|\bбелорус\w*|\bлукаш\w*|\blukash\w*|\bминск|\bminsk|\bтиханов\w*|\btsikhan)", re.IGNORECASE | re.UNICODE),
    "UA": re.compile(r"(\bукраин\w*|\bukrain\w*|\bкиев|\bkyiv|\bzelensk\w*|\bзеленск\w*|\bодесс|\bodes|\bхарьков|\bkharkiv|\bдонбас|\bdonbas|\bдонецк|\bdonetsk|\bкрым|\bcrimea|\bмариуполь|\bmariupol)", re.IGNORECASE | re.UNICODE),
    "RU": re.compile(r"(\bроссиян\w*|\brussian\b|\bnavalny\w*|\bнавальн\w*|\bпутин|\bputin|\bмосква|\bmoscow|\bкремль|\bkremlin)", re.IGNORECASE | re.UNICODE),
}
for c, pat in cs.items():
    df[c] = df["TITLE"].apply(lambda s: bool(pat.search(s))) | df["DESCRIPTION"].apply(lambda s: bool(pat.search(s)))

df["country_combo"] = df.apply(
    lambda r: "".join([k for k in "BUR" if {"B": r["BY"], "U": r["UA"], "R": r["RU"]}[k]]) or "(none)",
    axis=1,
)
combos = df["country_combo"].value_counts()

fig, ax = plt.subplots(figsize=(8, 3.5))
combos.plot.bar(ax=ax, color="#3a6ea5", edgecolor="white")
ax.set_title("Country-context combinations (B=Belarus, U=Ukraine, R=Russia)")
ax.set_ylabel("videos"); ax.tick_params(axis='x', rotation=0)
for i, v in enumerate(combos.values):
    ax.text(i, v + 80, f"{v:,}", ha="center", fontsize=8)
plt.tight_layout(); plt.show()
print(combos.to_string())


<a id="14-quality"></a>
## 14. Content-quality assessment

Three quality red flags — empty descriptions, near-empty descriptions,
and descriptions that are essentially only URLs/hashtags — together
account for about 1.4% of the subset. These videos will produce
low-information embeddings and should either be filtered or treated
with title-only encoding.


In [ ]:
df["desc_len"] = df["DESCRIPTION"].str.len()

def is_only_urls(s):
    if not s.strip(): return False
    no_url     = re.sub(r"https?://\S+", " ", s).strip()
    no_hashtag = re.sub(r"#\S+", "", no_url).strip()
    return len(no_hashtag) < 20

df["desc_only_urls"] = df["DESCRIPTION"].apply(is_only_urls)

print(f"Empty descriptions:               {(df.desc_len == 0).sum():,}")
print(f"Desc < 50 chars:                  {(df.desc_len < 50).sum():,}")
print(f"Desc essentially only URLs/tags:  {df.desc_only_urls.sum():,}")
print(f"Total quality red flags (any):    {((df.desc_len < 50) | df.desc_only_urls).sum():,}  ({100*((df.desc_len < 50) | df.desc_only_urls).mean():.1f}%)")


<a id="15-embedding-readiness"></a>
## 15. Embedding readiness — recommendation

For each video we construct a `feed_text` field as
`TITLE + " . " + DESCRIPTION` with URLs replaced by `<URL>` and excessive
whitespace collapsed. After this normalisation:

- 98.7% of videos have at least 30 meaningful (non-URL, non-hashtag)
  characters
- 94.3% have at least 100 meaningful characters
- median feed length is ~800 chars, 90th percentile is ~1,950 chars

This is well within the 512-token context window of `multilingual-e5-large`
or `BAAI/bge-m3` (a 512-token window holds roughly 2,000–2,500 characters
for Latin and 1,200–1,500 for Cyrillic). For the long-tail 1% above
4,500 chars, a simple truncation to 2,000 chars is recommended; richer
splitting can be added later if needed.


In [ ]:
def build_feed(row):
    t = row["TITLE"] or ""
    d = row["DESCRIPTION"] or ""
    s = t + " . " + d
    s = re.sub(r"https?://\S+", "<URL>", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["feed_text"] = df.apply(build_feed, axis=1)
df["feed_len"]  = df["feed_text"].str.len()

def meaningful_chars(s):
    s = re.sub(r"https?://\S+", "", s)
    s = re.sub(r"#\S+", "", s)
    return len(s.strip())

df["meaningful_len"] = df["feed_text"].apply(meaningful_chars)

print("feed_text length (chars):")
print(df["feed_len"].describe(percentiles=[.1,.5,.9,.99]).round(0).to_string())
print(f"\n≥100 meaningful chars: {(df.meaningful_len >= 100).sum():,} ({100*(df.meaningful_len >= 100).mean():.1f}%)")
print(f"≥30  meaningful chars: {(df.meaningful_len >= 30).sum():,}  ({100*(df.meaningful_len >= 30).mean():.1f}%)")

fig, ax = plt.subplots(figsize=(9, 3.4))
ax.hist(df["feed_len"].clip(upper=4000), bins=50, color="#3a6ea5", edgecolor="white")
ax.axvline(2000, color="#a55e3a", linestyle="--", label="recommended truncation (2000c)")
ax.set_title("feed_text length (clipped at 4000)")
ax.set_xlabel("characters"); ax.legend()
plt.tight_layout(); plt.show()


In [ ]:
# Save the prepared feed dataframe for the next stage (embedding)
out_dir = "outputs"
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "v2_feed_ready.csv.gz")

cols = ["YT_ID", "PUBLISHED_AT", "DEFAULT_AUDIO_LANG", "DURATION", "dur_sec",
        "LIVE_BROADCAST", "TITLE", "DESCRIPTION", "feed_text", "meaningful_len",
        "n_urls", "n_title_tags", "n_desc_tags",
        "protest_anywhere", "protest_in_title", "protest_in_desc",
        "BY", "UA", "RU", "country_combo",
        "desc_only_urls", "metadata_captured_on"]
df[cols].to_csv(out_path, index=False, compression='gzip')
print(f"Wrote {out_path}  ({os.path.getsize(out_path) / 1024 / 1024:.1f} MB)")


<a id="16-findings"></a>
## 16. Findings and next steps

### 16.1 Findings

1. **The text-ready subset is 21,710 videos** — small relative to the
   401K-video corpus, but adequate for embedding-based protest
   detection. The bottleneck is the 46-day metadata-fetch window,
   not the encoder or the storage.
2. **Coverage is skewed toward 2024**: 57% of the subset was published
   in 2024 alone. Earlier protest events (Belarus 2020, Maidan 2014)
   are dramatically underrepresented and require additional metadata
   fetches to be embeddable at scale.
3. **Languages require a multilingual encoder.** Russian dominates
   (~43% audio language), with Ukrainian, English, German and Belarusian
   all material. Twelve percent of titles mix Cyrillic and Latin scripts
   in the same string.
4. **Protest is named explicitly in only 9.4% of videos.** A regex over
   protest-root vocabulary catches 2,034 videos directly, but the
   remaining 90% of the Politics-topic subset is politically adjacent
   without surface protest keywords. This is the polysemy gap that
   embedding-based retrieval is meant to close.
5. **Country-context combinations differ systematically.** Russia-only
   and Ukraine+Russia together account for nearly half the subset.
   Belarus-only is a distinct pocket; Belarus+Ukraine+Russia is rare
   (~5%). The combination label is a stronger polysemy disambiguator
   than any single tag.
6. **Description boilerplate clusters videos by channel.** 612 videos
   share a 200-character description head with at least one other
   video, providing a free channel-id proxy until real channel
   identifiers are added to the data model.
7. **URLs dominate descriptions but carry no semantic value.** Median
   5 URLs per description; top domain is `bit.ly`. A `<URL>` placeholder
   replacement is recommended before encoding.
8. **Quality is good for embedding.** 98.7% of videos have ≥30
   meaningful characters after URL/hashtag stripping; median feed
   length is well within the 512-token context of `multilingual-e5-large`.

### 16.2 Recommended next steps

1. **Embed the 21,710-video subset** with `intfloat/multilingual-e5-large`
   on MPS (M2). Estimated time on a 16 GB M2: 15–25 minutes.
2. **Index in LanceDB or Chroma** (embedded vector store; no service
   required).
3. **Define anchor queries** for Belarus-2020, Russia-anti-war,
   Ukraine-Maidan, and so on. Retrieve top-K per anchor and inspect
   for polysemy resolution.
4. **Run a comparison** between regex-based protest detection (this
   notebook) and embedding-based retrieval. Report recall at top-K
   and precision on a small manually-labelled validation sample.
5. **For the broader 401K corpus**, schedule a metadata re-fetch via
   YouTube API (cost: ~22K video.list units = ~3 days at default
   10 K/day quota, or one paid burst). Title/description for the full
   corpus unlocks all of the v2 plan.

### 16.3 Outputs

- `outputs/v2_feed_ready.csv.gz` — feed dataframe with `feed_text`,
  protest/country flags, language and duration features. Ready for
  the embedding stage.

---

This notebook is the EDA companion to EXPORT 1 of the v2 metadata
extraction. Subsequent notebooks will build on `v2_feed_ready.csv.gz`
to implement the embedding stage and the anchor-based retrieval
pipeline.
